# Lezione 12 — Progetto integratore e ponte verso i transformer

**Corso:** NLP Classico con Python · **Blocco 5:** Chiusura
**Tempo stimato:** 120 minuti (la più densa del corso: è quella di chiusura) · **Capitolo del manuale:** `manual/12_proyecto_puente_transformers_IT.md`

## Cosa vedrai in questa lezione

- Una *pipeline* completa di classificazione di recensioni (TF-IDF + regressione logistica), riutilizzando le impostazioni delle lezioni 5 e 6.
- Un modello *transformer* pre-addestrato (`nlptown/bert-base-multilingual-uncased-sentiment`) applicato in **zero-shot**, senza adattare nulla al corpus del corso.
- Il confronto reale tra i due, sullo stesso insieme di test: accuratezza, F1, tempo di inferenza e costo computazionale.
- Un esempio di errore concreto di ciascun metodo, per confrontare quanto si può spiegare ognuno.
- Il ponte (senza insegnarlo) verso l'architettura dei transformer: attenzione, pre-addestramento e affinamento.

**Requisiti:** classificazione di testi con TF-IDF e regressione logistica (lezione 5); analisi del sentimento sulle recensioni (lezione 6). **Librerie:** `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `transformers`, `torch`. **Dati:** `datos/resenas_es.csv`, `datos/resenas_it.csv` e `datos/plan_b_transformer_resenas.json` (predizioni precalcolate, in caso di assenza di connessione); in Colab, carica la cartella `datos/`. **Tutte le cifre sono un esempio illustrativo**, su un corpus sintetico di 300 recensioni per lingua. **Durata dell'esecuzione:** pochi secondi se il modello è già in cache; se va scaricato (≈669 MB), qualche minuto la prima volta.

> **Nota:** questa lezione è più densa delle precedenti perché chiude il corso con un progetto integratore. Se preferisci dividerla in due sessioni, un taglio naturale è dopo la sezione 3 (prima di misurare tempi ed errori).

---
## Cella 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("torch", "torch")
asegurar("transformers", "transformers")
print("Librerie pronte.")

In [ ]:
import os
import time
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

IDIOMAS = ("es", "it")   # dato-es

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

MODELO_HF = "nlptown/bert-base-multilingual-uncased-sentiment"

def estrellas_a_etiqueta(etiqueta_estrellas):
    """Traduce l'output del modello ("1 star".."5 stars") in positivo/negativo/neutro con la regola 1-2->negativo, 3->neutro, 4-5->positivo."""
    n = int(etiqueta_estrellas[0])
    if n <= 2:
        return "negativo"
    if n == 3:
        return "neutro"
    return "positivo"

def particion(idioma):
    """Carica le recensioni e restituisce la stessa partizione 70/30 stratificata (seme 42) che usa tutto il capitolo."""
    df = pd.read_csv(cargar_datos(f"resenas_{idioma}.csv"))
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    return df, ent, prueba

def pipeline_clasico():
    """La stessa configurazione di TF-IDF + regressione logistica delle lezioni 5 e 6."""
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
                      ("clf", LogisticRegression(max_iter=1000))])

def cargar_plan_b():
    """Carica le predizioni precalcolate del transformer (piano B, senza connessione)."""
    with open(cargar_datos("plan_b_transformer_resenas.json"), encoding="utf-8") as f:
        return json.load(f)

def predecir_transformer(clasificador_hf, prueba, idioma, plan_b):
    """Se il modello è disponibile, predice davvero e cronometra; altrimenti usa le predizioni e il tempo precalcolati del piano B, allineati per id."""
    ids = prueba["id"].astype(str).tolist()
    datos_idioma = plan_b["idiomas"][idioma]
    if clasificador_hf is not None:
        t0 = time.perf_counter()
        salidas = clasificador_hf(prueba["texto"].tolist(), truncation=True, max_length=128)
        tiempo = time.perf_counter() - t0
        estrellas = [s["label"] for s in salidas]
    else:
        estrellas_por_id = dict(zip([str(x) for x in datos_idioma["id"]], datos_idioma["estrellas"]))
        estrellas = [estrellas_por_id[i] for i in ids]
        tiempo = datos_idioma["t_inferencia_transformer_total_s"]
    etiquetas = [estrellas_a_etiqueta(e) for e in estrellas]
    return estrellas, etiquetas, tiempo

print("Funzioni ausiliarie caricate.")

---
## 1. Il classico: TF-IDF + regressione logistica

Riutilizziamo la configurazione delle lezioni 5 e 6 (`TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)` + `LogisticRegression`), addestrata solo con il 70% di ogni lingua e valutata sul restante 30% (seme 42, la stessa partizione per tutto il capitolo).

In [ ]:
RESULT_CLASICO = {}
for idioma in IDIOMAS:
    df, ent, prueba = particion(idioma)
    pipe = pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred = pipe.predict(prueba["texto"])
    acc = accuracy_score(prueba["etiqueta"], pred)
    f1m = f1_score(prueba["etiqueta"], pred, average="macro")
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    RESULT_CLASICO[idioma] = dict(df=df, ent=ent, prueba=prueba, pipe=pipe, pred=list(pred), acc=acc, f1m=f1m, vocab=len(vocab))
    print(idioma, "| addestramento:", len(ent), "| test:", len(prueba),
          "| vocabolario:", len(vocab), "| accuratezza:", round(acc, 3), "| F1 macro:", round(f1m, 3))
    assert len(prueba) == 90 and len(ent) == 210          # 300 recensioni, partizione 70/30
    assert 1 / 3 < acc <= 1.0                               # meglio del caso (3 classi equilibrate)

### Cosa osservi?

Con solo 210 recensioni di addestramento (rumorose: refusi, maiuscole irregolari, allungamenti), il modello lineare recupera ben più di due terzi dei casi in entrambe le lingue, molto sopra il 33% del caso. È la stessa cifra, dello stesso ordine, della validazione incrociata documentata in `datos/LEEME.md`.

---
## 2. Il transformer pre-addestrato, in zero-shot

`nlptown/bert-base-multilingual-uncased-sentiment` (licenza MIT, ≈669 MB, 167 360 261 parametri) predice 1-5 stelle; le traduciamo nelle tre classi del corso con la regola **1-2→negativo, 3→neutro, 4-5→positivo** (giustificata nel capitolo, sezione 3.2). Il modello non vede mai una recensione del corpus del corso: è una valutazione **zero-shot**.

In [ ]:
assert estrellas_a_etiqueta("1 star") == "negativo" and estrellas_a_etiqueta("2 stars") == "negativo"
assert estrellas_a_etiqueta("3 stars") == "neutro"
assert estrellas_a_etiqueta("4 stars") == "positivo" and estrellas_a_etiqueta("5 stars") == "positivo"
print("Regola di mappatura verificata.")

La cella seguente **prova** a scaricare/caricare il modello reale; se non c'è rete o il download fallisce, prosegue con il **piano B** (predizioni precalcolate in `datos/plan_b_transformer_resenas.json`), senza interrompere il notebook.

In [ ]:
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()   # barra di avanzamento silenziata: cambia solo la presentazione, non il risultato
except Exception:
    pass
try:
    from transformers.utils import logging as _log_hf
    _log_hf.disable_progress_bar()
except Exception:
    pass
try:
    from transformers import pipeline as construir_pipeline_hf
    clasificador_hf = construir_pipeline_hf("sentiment-analysis", model=MODELO_HF)
    TRANSFORMER_OK = True
    print("Modello transformer caricato dalla cache di Hugging Face.")
except Exception as error:
    clasificador_hf = None
    TRANSFORMER_OK = False
    print("Non è stato possibile caricare il modello (senza rete o download fallito):", type(error).__name__)
    print("Si userà il PIANO B: predizioni precalcolate in datos/plan_b_transformer_resenas.json.")

PLAN_B = cargar_plan_b()
print("transformer live:", TRANSFORMER_OK)

In [ ]:
RESULT_TRAFO = {}
for idioma in IDIOMAS:
    prueba = RESULT_CLASICO[idioma]["prueba"]
    estrellas, pred, tiempo = predecir_transformer(clasificador_hf, prueba, idioma, PLAN_B)
    acc = accuracy_score(prueba["etiqueta"], pred)
    f1m = f1_score(prueba["etiqueta"], pred, average="macro")
    RESULT_TRAFO[idioma] = dict(estrellas=estrellas, pred=pred, acc=acc, f1m=f1m, tiempo=tiempo)
    print(idioma, "| accuratezza:", round(acc, 3), "| F1 macro:", round(f1m, 3),
          "| tempo totale (riferimento CPU):", PLAN_B["idiomas"][idioma]["t_inferencia_transformer_total_s"], "s per", len(prueba), "recensioni")
    assert len(pred) == len(prueba) == 90
    assert 1 / 3 < acc <= 1.0

### Cosa osservi?

Senza vedere una sola recensione del corpus del corso, il *transformer* pre-addestrato ottiene un'accuratezza più alta del classico addestrato specificamente su di esso, in entrambe le lingue (la sezione 3 del capitolo spiega perché: il dominio del modello — recensioni di prodotti — coincide con quello del corpus, e il classico ha avuto solo 210 esempi rumorosi da cui imparare).

---
## 3. Il confronto, faccia a faccia

Una tabella e un grafico con le quattro cifre (2 metodi × 2 lingue) sullo stesso insieme di test.

In [ ]:
filas = []
for idioma in IDIOMAS:
    filas.append(dict(idioma=idioma, metodo="clasico", exactitud=RESULT_CLASICO[idioma]["acc"], f1_macro=RESULT_CLASICO[idioma]["f1m"]))
    filas.append(dict(idioma=idioma, metodo="transformer", exactitud=RESULT_TRAFO[idioma]["acc"], f1_macro=RESULT_TRAFO[idioma]["f1m"]))
COMPARACION = pd.DataFrame(filas)
print(COMPARACION.round(3).to_string(index=False))
assert (COMPARACION.groupby("idioma")["exactitud"].apply(lambda s: s.iloc[1] > s.iloc[0])).all()   # in questo corpus, il transformer vince in entrambe le lingue

In [ ]:
fig, eje = plt.subplots(figsize=(7, 4))
ancho = 0.35
posiciones = np.arange(len(IDIOMAS))
for i, metodo in enumerate(("clasico", "transformer")):
    valores = [COMPARACION[(COMPARACION.idioma == idioma) & (COMPARACION.metodo == metodo)]["exactitud"].iloc[0] for idioma in IDIOMAS]
    eje.bar(posiciones + i * ancho, valores, ancho, label=metodo)
eje.set_xticks(posiciones + ancho / 2)
eje.set_xticklabels(IDIOMAS)
titulo = "Accuratezza per lingua e metodo"
eje.set_title(titulo)
eje.set_ylim(0, 1.0)
eje.legend()
plt.tight_layout(); plt.show()

### Cosa osservi?

Il *transformer* è avanti in entrambe le barre di ogni lingua. La differenza (≈0,13 in spagnolo, ≈0,06 in italiano) è reale in questo esperimento, ma ricorda l'avvertenza del capitolo: è il risultato di **questo** corpus piccolo e di **questo** dominio (recensioni di prodotto), non una legge generale.

---
## 4. Tempo di inferenza e costo computazionale (CPU, riferimento)

Misuriamo davvero il tempo di addestramento/predizione del classico e quello di predizione del *transformer*, e verifichiamo che il classico sia più veloce; perché il notebook dia lo stesso risultato stampato in qualunque esecuzione (i millisecondi esatti variano da una macchina o da un'esecuzione all'altra), mostriamo le **cifre di riferimento** registrate dall'autore su CPU, già validate da quella stessa verifica dal vivo.

In [ ]:
for idioma in IDIOMAS:
    ent, prueba, pipe = RESULT_CLASICO[idioma]["ent"], RESULT_CLASICO[idioma]["prueba"], RESULT_CLASICO[idioma]["pipe"]
    # Misuriamo davvero (per verificare l'ordine di grandezza); stampiamo le cifre di riferimento dell'autore (CPU), stabili tra le esecuzioni
    t0 = time.perf_counter(); pipeline_clasico().fit(ent["texto"], ent["etiqueta"]); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); pipe.predict(prueba["texto"]); t_pred = time.perf_counter() - t0
    tiempo_trafo = RESULT_TRAFO[idioma]["tiempo"]
    assert t_pred < tiempo_trafo   # il classico predice molto più veloce su CPU (verificato dal vivo)
    ref = PLAN_B["idiomas"][idioma]
    print(idioma, "| classico (riferimento CPU): addestrare", ref["t_entrenar_clasico_ref_s"], "s, predire", ref["t_predecir_clasico_ref_s"],
          f"s ({len(prueba)} recensioni) | transformer (riferimento CPU): predire", ref["t_inferencia_transformer_total_s"], "s")

IDIOMA_EJ = IDIOMAS[0]   # dato-es
n_params_trafo = PLAN_B["n_parametros"]
mb_trafo = PLAN_B["tamano_mb_fp32"]
n_params_clasico = 3 * RESULT_CLASICO[IDIOMA_EJ]["vocab"]
print("parametri del transformer:", n_params_trafo, f"(~{mb_trafo} MB, fp32) | licenza:", PLAN_B["licencia"])
print("coefficienti del classico, lingua di esempio:", IDIOMA_EJ, "|", n_params_clasico, "| rapporto:", round(n_params_trafo / n_params_clasico), "x")

### Cosa osservi?

Il classico predice in millisecondi (o meno) sulle 90 recensioni di test; il *transformer* impiega dell'ordine di un secondo per lo stesso lotto. E il *transformer* ha decine di migliaia di volte più parametri della regressione logistica di questo corpus. Nessuna cifra è "cattiva": il costo conta solo in base al volume e alle risorse disponibili (capitolo, sezione 7).

---
## 5. Facilità di spiegare gli errori

**Classico:** i coefficienti della regressione logistica si possono leggere parola per parola.

In [ ]:
pipe_ej = RESULT_CLASICO[IDIOMA_EJ]["pipe"]
vocab_ej = np.array(pipe_ej.named_steps["tfidf"].get_feature_names_out())
clf_ej = pipe_ej.named_steps["clf"]
for i, clase in enumerate(clf_ej.classes_):
    idx = np.argsort(-clf_ej.coef_[i])[:4]
    print(f"  {clase:>10}:", list(zip(vocab_ej[idx], np.round(clf_ej.coef_[i][idx], 2))))

In [ ]:
# Un errore concreto del classico: prima recensione di test mal classificata, in IDIOMA_EJ
prueba_ej = RESULT_CLASICO[IDIOMA_EJ]["prueba"].reset_index(drop=True)
pred_clasico_ej = RESULT_CLASICO[IDIOMA_EJ]["pred"]
errores_clasico = [i for i in range(len(prueba_ej)) if pred_clasico_ej[i] != prueba_ej["etiqueta"].iloc[i]]
i = errores_clasico[0]
print("testo:", prueba_ej["texto"].iloc[i])
print("reale:", prueba_ej["etiqueta"].iloc[i], "| predetto:", pred_clasico_ej[i])
print("spiegazione: si può sommare il coefficiente di ogni parola del testo e verificare il risultato (sezione 6.1 del capitolo)")
assert len(errores_clasico) > 0   # il classico non indovina il 100%

**Transformer:** non ci sono coefficienti per parola; cerchiamo comunque un errore concreto nell'insieme di test, per vederlo da vicino.

In [ ]:
pred_trafo_ej = RESULT_TRAFO[IDIOMA_EJ]["pred"]
estrellas_ej = RESULT_TRAFO[IDIOMA_EJ]["estrellas"]
errores_trafo = [i for i in range(len(prueba_ej)) if pred_trafo_ej[i] != prueba_ej["etiqueta"].iloc[i]]
j = errores_trafo[0]
print("testo:", prueba_ej["texto"].iloc[j])
print("reale:", prueba_ej["etiqueta"].iloc[j], "| predetto:", pred_trafo_ej[j], f"({estrellas_ej[j]})")
print("spiegazione: non c'è una tabella di pesi che lo giustifichi; solo l'output finale del modello (sezione 6.2 del capitolo)")
assert len(errores_trafo) > 0

### Cosa osservi?

L'errore del classico si spiega con numeri che puoi sommare a mano (i coefficienti della sezione precedente). L'errore del *transformer* si può solo descrivere ("ha predetto male"), non spiegare con la stessa precisione: non c'è una tabella equivalente senza strumenti aggiuntivi fuori dalla portata di questo corso.

---
## 6. Quando conviene ciascun approccio

| Situazione | Conviene |
|---|---|
| Pochi dati, dominio raro, bisogna spiegare ogni decisione | **Classico** |
| Risorse limitate (memoria, senza connessione, budget di calcolo) | **Classico** |
| Nessun dato proprio, dominio simile a quello di un modello già addestrato | **Transformer pre-addestrato** |
| Vocabolario reale molto vario (sinonimi, parafrasi, errori) | **Transformer** |
| Dati etichettati sufficienti e tempo per adattare il modello | **Affinare (*fine-tuning*) un transformer**, fuori da questo corso |

## 7. Ponte verso i transformer

Tutto ciò che il corso ha visto rappresenta il testo con un vettore per parola o coppia di parole, senza che il contesto lo cambi. I transformer aggiungono l'**attenzione**: ogni parola adatta la propria rappresentazione alle parole che la accompagnano in ogni frase concreta. Si costruiscono in due fasi: **pre-addestramento** (imparare da testo non etichettato, su grande scala) e **affinamento** (*fine-tuning*, specializzarsi in un compito con relativamente pochi esempi etichettati). Nessuna delle due si insegna in questo corso: è la porta d'ingresso al successivo.

---
## Riepilogo

- Una *pipeline* di NLP ha decisioni in ogni passo (preprocessing, rappresentazione, classificazione, valutazione, analisi degli errori) da giustificare, non da dare per scontate.
- Un modello *transformer* pre-addestrato si usa con `pipeline(...)` di Hugging Face in poche righe, senza addestrare nulla.
- In questo corpus (recensioni sintetiche, dominio di prodotti), il *transformer* in zero-shot **ha superato** il classico addestrato specificamente: un risultato reale, non forzato, con una spiegazione identificabile (capitolo, sezione 4).
- Il classico si spiega con coefficienti; il *transformer*, no, senza strumenti aggiuntivi: è una differenza pratica al momento di scegliere.
- I transformer aggiungono l'attenzione sulle rappresentazioni che già conosci; si costruiscono con pre-addestramento e affinamento, il tema del corso successivo.

**Questo chiude il corso.** Il notebook di pratica (`clase12_practica_proyecto_puente_transformers_IT.ipynb`) ripete il progetto su un taglio diverso dei dati e con più dettaglio, con esercizi passo passo.